# Getting Started with IONIS Datasets

This notebook demonstrates how to load and explore the IONIS propagation datasets.

**What you'll learn:**
- How to download datasets using `ionis-download`
- How to load datasets into pandas DataFrames
- Basic exploration and filtering
- Understanding the data schema

## 1. Setup

First, let's install the required packages and download sample data.

In [ ]:
# Install packages (skip if already installed)
# !pip install ionis-jupyter ionis-mcp

In [ ]:
# Download the minimal bundle (~430 MB)
# This includes: contest signatures, grid lookup, solar indices
# !ionis-download --bundle minimal

# For more data, use:
# !ionis-download --bundle recommended  # ~1.1 GB
# !ionis-download --bundle full         # ~15 GB (all datasets)

## 2. Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ionis_jupyter import (
    load_dataset,
    list_datasets,
    get_data_dir,
    grid_to_latlon,
    grid_distance,
)

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 3. Check Available Datasets

In [ ]:
# Show data directory location
print(f"Data directory: {get_data_dir()}")
print()

# List available datasets
datasets = list_datasets()
for name, info in datasets.items():
    status = "✓" if info["exists"] else "✗"
    rows = f"{info['rows']:,}" if info["rows"] else "N/A"
    print(f"{status} {name:12} {rows:>15} rows")

## 4. Load a Dataset

Let's load the contest signatures dataset (included in the minimal bundle).

In [ ]:
# Load contest signatures
contest = load_dataset("contest")
print(f"Loaded {len(contest):,} contest signatures")
print()
print("Columns:", list(contest.columns))

In [ ]:
# Preview the data
contest.head(10)

## 5. Understanding the Schema

Each signature represents an aggregated propagation observation:

| Column | Description |
|--------|-------------|
| `tx_grid_4` | Transmitter grid (4-char Maidenhead) |
| `rx_grid_4` | Receiver grid (4-char Maidenhead) |
| `band` | Band ID (102=160m, 107=20m, 111=10m, etc.) |
| `hour` | Hour of day (0-23 UTC) |
| `month` | Month (1-12) |
| `median_snr` | Median SNR in dB |
| `spot_count` | Number of observations in this bucket |
| `snr_std` | SNR standard deviation |
| `avg_sfi` | Average Solar Flux Index |
| `avg_kp` | Average Kp geomagnetic index |
| `avg_distance` | Average path distance (km) |
| `avg_azimuth` | Average bearing (degrees) |

In [ ]:
# Band ID reference
BAND_NAMES = {
    102: "160m", 103: "80m", 104: "60m", 105: "40m", 106: "30m",
    107: "20m", 108: "17m", 109: "15m", 110: "12m", 111: "10m",
}

# Show band distribution
band_counts = contest.groupby("band").size().sort_index()
for band_id, count in band_counts.items():
    name = BAND_NAMES.get(band_id, str(band_id))
    print(f"{name:>5}: {count:>10,} signatures")

## 6. Basic Filtering

In [ ]:
# Filter to 20m band
contest_20m = contest[contest["band"] == 107]
print(f"20m signatures: {len(contest_20m):,}")

# Filter to specific grids
# Example: Idaho (DN13) to Europe (JN48)
idaho_eu = contest[
    (contest["tx_grid_4"] == "DN13") & 
    (contest["rx_grid_4"].str.startswith("JN"))
]
print(f"Idaho to JN* paths: {len(idaho_eu):,}")

In [ ]:
# Filter by hour (daytime paths)
daytime = contest[(contest["hour"] >= 12) & (contest["hour"] <= 20)]
print(f"Daytime signatures (12-20 UTC): {len(daytime):,}")

# Filter by solar conditions (high SFI)
high_sfi = contest[contest["avg_sfi"] > 150]
print(f"High SFI (>150) signatures: {len(high_sfi):,}")

## 7. Grid Utilities

In [ ]:
# Convert grid to lat/lon
lat, lon = grid_to_latlon("DN13")
print(f"DN13 (Idaho): {lat:.2f}°N, {lon:.2f}°W")

lat, lon = grid_to_latlon("JN48")
print(f"JN48 (Central Europe): {lat:.2f}°N, {lon:.2f}°E")

In [ ]:
# Calculate distance between grids
distance = grid_distance("DN13", "JN48")
print(f"DN13 to JN48: {distance:,.0f} km")

## 8. Quick Visualization

In [ ]:
# SNR distribution by band
fig, ax = plt.subplots(figsize=(12, 6))

band_data = []
band_labels = []
for band_id in sorted(contest["band"].unique()):
    data = contest[contest["band"] == band_id]["median_snr"].dropna()
    if len(data) > 100:
        band_data.append(data.values)
        band_labels.append(BAND_NAMES.get(band_id, str(band_id)))

ax.boxplot(band_data, labels=band_labels)
ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Band")
ax.set_ylabel("Median SNR (dB)")
ax.set_title("SNR Distribution by Band — Contest Signatures")
plt.tight_layout()
plt.show()

In [ ]:
# Hour distribution
fig, ax = plt.subplots(figsize=(12, 5))

hour_counts = contest.groupby("hour").size()
ax.bar(hour_counts.index, hour_counts.values, color="steelblue", alpha=0.7)
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("Number of Signatures")
ax.set_title("Signature Distribution by Hour")
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

## 9. Next Steps

Now that you can load and explore the data, continue with:

- **02-band-openings.ipynb** — Hour×month heatmaps
- **03-solar-correlation.ipynb** — SFI effects on propagation
- **04-path-analysis.ipynb** — Deep dive on specific paths
- **05-greyline-terminator.ipynb** — Day/night boundary analysis
- **06-tid-detection.ipynb** — TID research techniques